# Dự báo Độ mặn Nước Bề mặt bằng Mô hình Học máy Tối ưu hóa PSO

Notebook này huấn luyện và đánh giá 5 mô hình:
1. Bagging Regressor
2. Random Forest (RF)
3. 1D-CNN (PyTorch)
4. PSO-Bagging
5. PSO-Random Forest (PSO-RF)

**Đánh giá**: R², RMSE, MAE, Explained Variance

In [1]:
# === Cell 1: Cài đặt thư viện ===
!pip install pyswarm shap torch scikit-learn pandas numpy matplotlib seaborn --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# === Cell 2: Import thư viện và định nghĩa hàm đánh giá ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import copy
import re
from datetime import datetime

from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from pyswarm import pso
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)


def calc_r2(y_true, y_pred):
    """Coefficient of Determination (R²)"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)


def calc_rmse(y_true, y_pred):
    """Root Mean Square Error (g/l)"""
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def calc_mae(y_true, y_pred):
    """Mean Absolute Error (g/l)"""
    return np.mean(np.abs(y_true - y_pred))


def calc_explained_variance(y_true, y_pred):
    """Explained Variance (%)"""
    return (1 - np.var(y_true - y_pred) / np.var(y_true)) * 100


def evaluate_model(y_true, y_pred):
    """Trả về dict chứa 4 metrics"""
    return {
        'R2': round(calc_r2(y_true, y_pred), 4),
        'RMSE': round(calc_rmse(y_true, y_pred), 4),
        'MAE': round(calc_mae(y_true, y_pred), 4),
        'Ex_VAR (%)': round(calc_explained_variance(y_true, y_pred), 2)
    }


print('Import và định nghĩa hàm đánh giá hoàn tất.')

C:\Users\nbbxv\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import và định nghĩa hàm đánh giá hoàn tất.


In [9]:
# === Cell 3: Tải dữ liệu ===
CSV_FILE = 'Check_BMA_Chuanhoa_LS8_20192020_159.csv'

if os.path.exists(CSV_FILE):
    print(f'Đọc dữ liệu từ file: {CSV_FILE}')
    df = pd.read_csv(CSV_FILE)
else:
    print(f"Khong tim thay du lieu tu file : {CSV_FILE}")   

pattern = r'^\d{2}Th\d{4}'
time_cols = [col for col in df.columns if re.match(pattern, col)]
print(f"Tim thay tong cong : {len(time_cols)} cot ngay")
target_col = 'SALINITY_OK'
rs_features = ["DEM","Lat","Long","EVI","NDBI","NDMI","NDSI","NDVI","NDWI","VSSI","SAVI","COSRI","EVI2",'MSI','ND23','ND47','SI1','SI2','SI3','SI4','SI5','SI6','SI7',"SI8",'SI9',"CA","B","G","R","NIR",'PCA1', 'PCA2','PCA3','PCA4','PCA5']

def parse_date_column(col_name):
    day = int(col_name[0:2])
    month = int(col_name[4:6])
    year = int('20' + col_name[6:8])
    return datetime(year, month, day)

def get_safe_date(row):
    active_cols = [ col for col in time_cols if row[col] == 1]
    
    if len(active_cols) == 1:
        return parse_date_column(active_cols[0])
    
    elif len(active_cols) > 1 :
        print(f"Co nhieu ngay quan trac chung 1 hang")
        return parse_date_column(active_cols[0]) # Lay ngay dau tien
    else:
        return pd.NaT

df["date"] = df.apply(get_safe_date, axis = 1)
df = df.dropna(subset = ["date"]).reset_index(drop = True)
df["doy"] = df["date"].dt.dayofyear
df["year"] = df["date"].dt.year
df["sin_doy"] = np.sin(2*np.pi*df["doy"]/ 365.25)
df["cos_doy"] = np.cos(2*np.pi*df["doy"]/ 365.25)

print(df["date"].dt.month)

Đọc dữ liệu từ file: Check_BMA_Chuanhoa_LS8_20192020_159.csv
Tim thay tong cong : 37 cot ngay
0      1
1      1
2      1
3      1
4      1
      ..
505    6
506    6
507    6
508    6
509    6
Name: date, Length: 510, dtype: int32


In [23]:
# Cell 4: chia tap train test 
df_train = df[(df['date'] >= '2019-01-01') & (df['date'] < '2020-03-01')].reset_index(drop= True)

df_test = df[(df['date'] >= '2020-03-01')].reset_index(drop=True)
X_train_base = df_train[rs_features + ['sin_doy', "cos_doy"]].values
X_test_base = df_test[rs_features + ['sin_doy', "cos_doy"]].values

X_train_st = df_train[['Lat', 'Long', 'doy']].values
X_test_st = df_test[['Lat', 'Long', 'doy']].values

y_train = df_train['SALINITY_OK'].values
y_test = df_test['SALINITY_OK'].values

print(f"tap train co so mau la {X_train_base.shape[0]}, do man trung binh: {y_train.mean()}")
print(f"tap test co so mau la {X_test_base.shape[0]}, do man trung binh la {y_test.mean()}")
print(f"so dac trung dua vao mo hinh cay la {X_train_base.shape[1]}")
print(f"so dac trung dua vao mo hinh GP la {X_train_st.shape[1]}")



tap train co so mau la 359, do man trung binh: 5.2808356545961
tap test co so mau la 151, do man trung binh la 8.549006622516558
so dac trung dua vao mo hinh cay la 37
so dac trung dua vao mo hinh GP la 3


In [ ]:
# === Cell 5: Định nghĩa 3 mô hình cơ sở (Base Models) ===

# --- 1. Bagging Regressor ---
bagging_model = BaggingRegressor(
    estimator=DecisionTreeRegressor(random_state=42),
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# --- 2. Random Forest Regressor ---
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# --- 3. 1D-CNN (PyTorch) với BatchNorm và Dropout ---
class CNN1D(nn.Module):
    def __init__(self, n_features):
        super(CNN1D, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.3)
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(32)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.3)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * n_features, 64)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.dropout1(self.relu1(self.bn1(self.conv1(x))))
        x = self.dropout2(self.relu2(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = self.dropout3(self.relu3(self.fc1(x)))
        x = self.fc2(x)
        return x


def train_cnn(X_train, y_train, X_test, y_test, n_features, epochs=300, lr=0.001, batch_size=32):
    """Huấn luyện mô hình 1D-CNN với early stopping và learning rate scheduler"""
    X_tr = torch.FloatTensor(X_train).unsqueeze(1)
    y_tr = torch.FloatTensor(y_train).unsqueeze(1)
    X_te = torch.FloatTensor(X_test).unsqueeze(1)
    y_te = torch.FloatTensor(y_test).unsqueeze(1)

    train_dataset = TensorDataset(X_tr, y_tr)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = CNN1D(n_features)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

    best_loss = float('inf')
    best_model_state = None
    patience = 40
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_te)
            val_loss = criterion(val_pred, y_te).item()

        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if (epoch + 1) % 50 == 0:
            print(f'  Epoch [{epoch+1}/{epochs}], Train Loss: {epoch_loss/len(train_loader):.4f}, Val Loss: {val_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        train_pred = model(X_tr).numpy().flatten()
        test_pred = model(X_te).numpy().flatten()

    return model, train_pred, test_pred


print('Định nghĩa 3 mô hình cơ sở hoàn tất.')
print('  1. Bagging Regressor')
print('  2. Random Forest Regressor')
print('  3. 1D-CNN (PyTorch, BatchNorm + Dropout + EarlyStopping)')

In [ ]:
# === Cell 6: Tối ưu hóa PSO cho Bagging và Random Forest ===

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

def pso_objective_bagging(params):
    """Hàm mục tiêu PSO cho Bagging: tối thiểu hóa MAE qua cross-validation"""
    n_estimators = int(params[0])
    max_depth = int(params[1])
    min_samples_split = int(params[2])
    min_samples_leaf = int(params[3])
    max_features_frac = params[4]
    max_samples_frac = params[5]

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('bagging', BaggingRegressor(
            estimator=DecisionTreeRegressor(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                random_state=42
            ),
            n_estimators=n_estimators,
            max_features=max_features_frac,
            max_samples=max_samples_frac,
            random_state=42,
            n_jobs=-1
        ))
    ])
    # X_train (chưa chuẩn hóa) được đưa vào, Pipeline tự fit_transform trên mỗi fold
    scores = cross_val_score(model, X_train_base, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)


def pso_objective_rf(params):
    """Hàm mục tiêu PSO cho RF: tối thiểu hóa MAE qua cross-validation"""
    n_estimators = int(params[0])
    max_depth = int(params[1])
    min_samples_split = int(params[2])
    min_samples_leaf = int(params[3])
    max_features_frac = params[4]

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features_frac,
            random_state=42,
            n_jobs=-1
        ))
    ])
    scores = cross_val_score(model, X_train_base, y_train, cv=5, scoring='neg_mean_absolute_error')
    return -np.mean(scores)


# Giới hạn siêu tham số cho Bagging:
# [n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, max_samples]
lb_bag = [50, 3, 2, 2, 0.4, 0.5]
ub_bag = [300, 10, 10, 10, 0.9, 0.9]

# Giới hạn siêu tham số cho RF:
# [n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features]
lb_rf = [50, 3, 2, 2, 0.3]
ub_rf = [300, 10, 10, 10, 0.9]

def extract_pso_params(pso_res):
    # Xử lý mọi cấu trúc trả về có thể có của thư viện pso
    if isinstance(pso_res, tuple):
        params = pso_res[0]
        score = pso_res[1] if len(pso_res) > 1 else None
    elif hasattr(pso_res, 'x') and hasattr(pso_res, 'fun'):
        params = pso_res.x
        score = pso_res.fun
    else:
        params = pso_res
        score = None
        
    # Ép kiểu thành list nếu nó là dict để tránh KeyError: 0
    if isinstance(params, dict):
        params = list(params.values())
    elif not isinstance(params, (list, np.ndarray)):
        params = list(params)
    return params, score

# --- PSO cho Bagging ---
print('=== PSO tối ưu Bagging Regressor ===')
pso_result_bag = pso(pso_objective_bagging, lb_bag, ub_bag, swarmsize=30, maxiter=50, debug=True)
best_params_bag, best_score_bag = extract_pso_params(pso_result_bag)
print(f'\nBagging - Siêu tham số tối ưu:')
print(f'  n_estimators      = {int(best_params_bag[0])}')
print(f'  max_depth         = {int(best_params_bag[1])}')
print(f'  min_samples_split = {int(best_params_bag[2])}')
print(f'  min_samples_leaf  = {int(best_params_bag[3])}')
print(f'  max_features      = {best_params_bag[4]:.3f}')
print(f'  max_samples       = {best_params_bag[5]:.3f}')
print(f'  Best CV MAE       = {best_score_bag:.4f}\n')

# --- PSO cho Random Forest ---
print('=== PSO tối ưu Random Forest ===')
pso_result_rf = pso(pso_objective_rf, lb_rf, ub_rf, swarmsize=30, maxiter=50, debug=True)
best_params_rf, best_score_rf = extract_pso_params(pso_result_rf)
print(f'\nRF - Siêu tham số tối ưu:')
print(f'  n_estimators      = {int(best_params_rf[0])}')
print(f'  max_depth         = {int(best_params_rf[1])}')
print(f'  min_samples_split = {int(best_params_rf[2])}')
print(f'  min_samples_leaf  = {int(best_params_rf[3])}')
print(f'  max_features      = {best_params_rf[4]:.3f}')
print(f'  Best CV MAE       = {best_score_rf:.4f}\n')

# --- Tạo mô hình PSO-Bagging với siêu tham số tối ưu ---
pso_bagging_model = Pipeline([
    ('scaler', StandardScaler()),
    ('bagging', BaggingRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=int(best_params_bag[1]),
            min_samples_split=int(best_params_bag[2]),
            min_samples_leaf=int(best_params_bag[3]),
            random_state=42
        ),
        n_estimators=int(best_params_bag[0]),
        max_features=best_params_bag[4],
        max_samples=best_params_bag[5],
        random_state=42,
        n_jobs=-1
    ))
])

# --- Tạo mô hình PSO-RF với siêu tham số tối ưu ---
pso_rf_model = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(
        n_estimators=int(best_params_rf[0]),
        max_depth=int(best_params_rf[1]),
        min_samples_split=int(best_params_rf[2]),
        min_samples_leaf=int(best_params_rf[3]),
        max_features=best_params_rf[4],
        random_state=42,
        n_jobs=-1
    ))
])

print('Tạo mô hình PSO-Bagging và PSO-RF hoàn tất.')

In [ ]:
# === Cell 7: Huấn luyện toàn bộ 5 mô hình & Tổng hợp bảng so sánh ===

results = []

# --- 1. Bagging Regressor ---
print('Huấn luyện Bagging Regressor...')
bagging_model.fit(X_train_base, y_train)
bag_train_pred = bagging_model.predict(X_train_base)
bag_test_pred = bagging_model.predict(X_test_base)
results.append({
    'Model': 'Bagging',
    **{f'Train_{k}': v for k, v in evaluate_model(y_train, bag_train_pred).items()},
    **{f'Test_{k}': v for k, v in evaluate_model(y_test, bag_test_pred).items()}
})
print('  Bagging hoàn tất.')

# --- 2. Random Forest ---
print('Huấn luyện Random Forest...')
rf_model.fit(X_train_scaled, y_train)
rf_train_pred = rf_model.predict(X_train_scaled)
rf_test_pred = rf_model.predict(X_test_scaled)
results.append({
    'Model': 'Random Forest',
    **{f'Train_{k}': v for k, v in evaluate_model(y_train, rf_train_pred).items()},
    **{f'Test_{k}': v for k, v in evaluate_model(y_test, rf_test_pred).items()}
})
print('  Random Forest hoàn tất.')

# --- 3. 1D-CNN ---
print('Huấn luyện 1D-CNN...')
n_features = X_train_scaled.shape[1]
cnn_model, cnn_train_pred, cnn_test_pred = train_cnn(
    X_train_scaled, y_train, X_test_scaled, y_test,
    n_features=n_features, epochs=300, lr=0.001, batch_size=32
)
results.append({
    'Model': '1D-CNN',
    **{f'Train_{k}': v for k, v in evaluate_model(y_train, cnn_train_pred).items()},
    **{f'Test_{k}': v for k, v in evaluate_model(y_test, cnn_test_pred).items()}
})
print('  1D-CNN hoàn tất.')

# --- 4. PSO-Bagging ---
print('Huấn luyện PSO-Bagging...')
pso_bagging_model.fit(X_train, y_train)
pso_bag_train_pred = pso_bagging_model.predict(X_train)
pso_bag_test_pred = pso_bagging_model.predict(X_test)
results.append({
    'Model': 'PSO-Bagging',
    **{f'Train_{k}': v for k, v in evaluate_model(y_train, pso_bag_train_pred).items()},
    **{f'Test_{k}': v for k, v in evaluate_model(y_test, pso_bag_test_pred).items()}
})
print('  PSO-Bagging hoàn tất.')

# --- 5. PSO-RF ---
print('Huấn luyện PSO-RF...')
pso_rf_model.fit(X_train, y_train)
pso_rf_train_pred = pso_rf_model.predict(X_train)
pso_rf_test_pred = pso_rf_model.predict(X_test)
results.append({
    'Model': 'PSO-RF',
    **{f'Train_{k}': v for k, v in evaluate_model(y_train, pso_rf_train_pred).items()},
    **{f'Test_{k}': v for k, v in evaluate_model(y_test, pso_rf_test_pred).items()}
})
print('  PSO-RF hoàn tất.')

# --- Bảng tổng hợp hiệu năng ---
print('\n' + '='*80)
print('BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH')
print('='*80)
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
results_df

In [ ]:
# === Cell 8: Biểu đồ dự báo & SHAP ===

# --- 8a. Scatter Plot: Giá trị thực tế vs Dự báo (PSO-RF) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Training set
axes[0].scatter(y_train, pso_rf_train_pred, alpha=0.6, edgecolors='k', linewidths=0.5, c='#2196F3')
min_val = min(y_train.min(), pso_rf_train_pred.min())
max_val = max(y_train.max(), pso_rf_train_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='1:1 Line')
train_metrics = evaluate_model(y_train, pso_rf_train_pred)
axes[0].set_xlabel('Giá trị thực tế (g/l)', fontsize=12)
axes[0].set_ylabel('Giá trị dự báo (g/l)', fontsize=12)
axes[0].set_title(f'PSO-RF - Training Set\nR²={train_metrics["R2"]}, RMSE={train_metrics["RMSE"]}', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Testing set
axes[1].scatter(y_test, pso_rf_test_pred, alpha=0.6, edgecolors='k', linewidths=0.5, c='#FF9800')
min_val = min(y_test.min(), pso_rf_test_pred.min())
max_val = max(y_test.max(), pso_rf_test_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='1:1 Line')
test_metrics = evaluate_model(y_test, pso_rf_test_pred)
axes[1].set_xlabel('Giá trị thực tế (g/l)', fontsize=12)
axes[1].set_ylabel('Giá trị dự báo (g/l)', fontsize=12)
axes[1].set_title(f'PSO-RF - Testing Set\nR²={test_metrics["R2"]}, RMSE={test_metrics["RMSE"]}', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scatter_pso_rf.png', dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ Scatter Plot: scatter_pso_rf.png')

# --- 8b. SHAP Summary Plot (PSO-RF) ---
print('\nĐang tính SHAP values cho PSO-RF...')
explainer = shap.TreeExplainer(pso_rf_model.named_steps['rf'])
shap_values = explainer.shap_values(X_test_scaled)

X_test_df = pd.DataFrame(X_test_scaled, columns=selected_features)

# Bar plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title('SHAP Feature Importance - PSO-RF', fontsize=14)
plt.tight_layout()
plt.savefig('shap_summary_pso_rf.png', dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ SHAP bar: shap_summary_pso_rf.png')

# Beeswarm plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title('SHAP Summary Plot (Beeswarm) - PSO-RF', fontsize=14)
plt.tight_layout()
plt.savefig('shap_beeswarm_pso_rf.png', dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ SHAP beeswarm: shap_beeswarm_pso_rf.png')

print('\n' + '='*50)
print('HOÀN TẤT TOÀN BỘ PIPELINE')
print('='*50)